# Lesson 04 - Designmønster for brug af værktøjer

I denne lektion vil du lære designmønstret **Brug af værktøj** for AI-agenter ved brug af Microsoft Agent Framework (Python). Vi gennemgår:

- Definition af funktionsværktøjer med `@tool` dekoratoren og typede parametre
- Tilvejebringelse af værktøjs-skemaer, så modellen ved, hvad hvert værktøj gør
- Styring af værktøjsudførelse med `approval_mode`
- Returnering af **struktureret output** via Pydantic-modeller og `response_format`

Scenariet er en **rejsebookingsagent**, der kan søge efter destinationer, tjekke tilgængelighed og hente flyinformation.


## Opsætning


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definere værktøjer med @tool-dekoratøren

`@tool`-dekoratøren omdanner en almindelig Python-funktion til et værktøj, som en agent kan kalde.
Vigtige punkter:

- **Docstringen** bliver til værktøjets beskrivelse, som modellen ser.
- **Typeannoteringer** (inklusive `Annotated` med beskrivelser) definerer værktøjets skema.
- `approval_mode` styrer, om brugeren skal godkende hvert kald, før det udføres.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Oprettelse af en Agent med Flere Værktøjer

Overgiv alle tre værktøjer til klienten, så modellen kan bruge dem, den har brug for, for at svare på brugerens spørgsmål.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Struktureret output med værktøjer

Ved at sætte `response_format` til en Pydantic-model, bliver agenten tvunget til at returnere et veltypet JSON-objekt i stedet for fri tekst. Dette er nyttigt, når efterfølgende kode skal kunne anvende resultatet programmatisk.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Mønstre for godkendelse af værktøj

`approval_mode`-parameteren på `@tool` styrer, om værktøjskald kræver menneskelig godkendelse, før de udføres:

| Mode | Adfærd |
|---|---|
| `"never_require"` | Værktøjet kører automatisk — ingen brugerbekræftelse nødvendig. |
| `"always_require"` | Hvert kald skal godkendes af brugeren, før det udføres. |

Brug `"always_require"` for værktøjer, der har bivirkninger (f.eks. booking af fly, opkrævning på kreditkort), så et menneske forbliver involveret.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

I denne lektion lærte du at:

1. **Definere værktøjer** ved hjælp af `@tool` dekoratøren med typerede parametre og docstrings, der fungerer som værktøjets skema.
2. **Sammensætte flere værktøjer**, så agenten kan kalde dem i rækkefølge for at besvare komplekse forespørgsler.
3. **Returnere struktureret output** ved at sende en Pydantic-model som `response_format`.
4. **Kontrollere godkendelse af værktøjer** med `approval_mode` for at holde et menneske involveret i følsomme operationer.

Disse mønstre udgør grundlaget for at bygge pålidelige, produktionsklare agenter, der sikkert kan interagere med eksterne systemer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiske oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for eventuelle misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
